# TV Advertising Performance Analysis  
### Sample Media Advertiser — Business Intelligence Case Study

> **Portfolio note:** This notebook is a sanitized version prepared for public portfolio review. Original source data and case-specific files are excluded. The workflow demonstrates KPI engineering, traffic-response attribution preparation, and Tableau-ready dataset generation using generalized naming.

## 1. Objective

The objective of this analysis is to evaluate the effectiveness of Sample Media Advertiser’s TV advertising campaigns by measuring cost efficiency across different networks.

Specifically, the analysis aims to:
- Calculate key performance metrics (CPM, CPV, CR, CPA)
- Identify the most and least cost-efficient TV networks
- Provide recommendations on budget allocation
- Validate TV attribution using site traffic data

The final deliverable is a Tableau dashboard designed for recurring monthly reporting.

## 2. Data Sources

This analysis uses four datasets:

- **Spots (TV Airings):** Spot-level data including airing time, network, spend, impressions, and lift  
- **Spots Aggregated by Day:** Daily aggregated metrics by network (spend, impressions, lift)  
- **Purchase Exit Survey:** Daily TV-attributed purchases by network based on customer responses  
- **Site Traffic Data:** Minute-level site traffic with baseline estimates  

These datasets are combined to evaluate performance and validate TV attribution.

## 3. Data Loading

The datasets are loaded from the provided Excel file. Each sheet corresponds to a different data source used in the analysis.

- TV airings data (spot-level)
- Aggregated daily advertising data (network × day)
- Purchase exit survey data (network × day)
- Site traffic data (minute-level)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

file_name = list(uploaded.keys())[0]

spots = pd.read_excel(file_name, sheet_name="spots")
spots_daily = pd.read_excel(file_name, sheet_name="spots_agg_day")
survey = pd.read_excel(file_name, sheet_name="purchase exit survey data")
traffic = pd.read_excel(file_name, sheet_name="traffic")

In [ ]:
print("Spots:", spots.shape)
print("Daily Aggregated:", spots_daily.shape)
print("Survey:", survey.shape)
print("Traffic:", traffic.shape)

## 4. Initial Data Understanding

Before performing data cleaning and metric calculations, it is important to understand the structure and granularity of each dataset.

- The airings data contains spot-level advertising activity, including spend, impressions, and lift.
- The aggregated daily data summarizes performance at the network and day level.
- The purchase exit survey data captures TV-attributed purchases by network and day, based on customer responses.
- The traffic data provides minute-level site activity along with a baseline estimate, which can be used to measure incremental lift.

From an initial review, several potential data challenges are identified:

- Network names may not be consistently labeled across datasets, which could impact joins and aggregations
- Some survey responses have missing or “other” network values, limiting attribution accuracy
- There may be cases where spend exists without purchases, or purchases exist without corresponding spend
- Lift values can be negative, which may affect downstream metric calculations

These issues will be addressed in the data cleaning stage to ensure accurate and reliable performance measurement.

## 5. Data Cleaning

To ensure accurate and reliable metric calculations, several data cleaning steps are performed. These steps focus on preparing key fields for reliable joins across datasets, handling ambiguous values in attribution data, and ensuring that calculated metrics are interpretable and robust for recurring reporting.

### 5.1 Network Name Standardization

Network names are standardized across datasets to ensure consistent joins between advertising and survey data.

Although the data appears largely consistent, this step is applied as a best practice to prevent potential mismatches caused by variations in casing or whitespace, especially in a recurring reporting context.

This ensures that network-level aggregations and performance metrics are computed accurately.

In [ ]:
# standardize network names (lowercase + strip spaces)

spots_daily['network_name'] = spots_daily['network_name'].str.lower().str.strip()
survey['network_name'] = survey['network_name'].str.lower().str.strip()

### 5.2 Handling Missing and "Other" Network Values

Some survey responses contain missing or "other" network values, which limit precise attribution to specific networks.

Instead of removing these records, they are retained to preserve overall data integrity. A flag is created to distinguish records that can be reliably attributed at the network level.

This approach ensures that network-level performance metrics are calculated using only valid attribution data, while overall metrics remain representative.

In [ ]:
# create a clean flag for valid networks

survey['valid_network'] = ~survey['network_name'].isin(['', 'other']) & survey['network_name'].notna()

### 5.3 Handling Negative Lift Values

Lift represents incremental traffic above a baseline estimate. In some cases, negative lift values may occur due to measurement noise or natural variation.

Since negative lift does not represent meaningful incremental impact, these values are set to zero for metric calculations. This prevents distortion in CPV and conversion rate calculations.

This approach preserves the original records while ensuring that derived metrics remain interpretable.

In [ ]:
# set negative lift to zero

spots_daily['lift'] = spots_daily['lift'].apply(lambda x: max(x, 0))

### 5.4 Date Standardization

Date fields are converted to a consistent datetime format to enable accurate time-based aggregation.

In addition, a monthly period field is created to support reporting at the required granularity, including overall, monthly, and network-by-month analyses.

This ensures that the dataset is properly structured for recurring reporting and dashboard integration.

In [ ]:
# extract month for later aggregation
spots_daily['month'] = spots_daily['day'].dt.to_period('M')
survey['month'] = survey['day'].dt.to_period('M')

In addition, a monthly period field is created to support time-based aggregation in later analysis.

## 6. Performance Metric Calculation

After cleaning and standardizing the datasets, performance metrics are calculated to evaluate TV advertising effectiveness across networks and time periods.

Because these metrics are ratio-based, the data is first aggregated to the required reporting level before calculating CPM, CPV, conversion rate (CR), and cost per acquisition (CPA). This ensures that the reported values accurately reflect overall performance.

### 6.1 Build the Analysis Table

To calculate performance metrics, the advertising data and survey data are aggregated to a common reporting level and then merged.

At this stage, monthly network-level summaries are created by combining:
- advertising performance data (spend, impressions, lift)
- TV-attributed purchases from the exit survey

In [ ]:
# aggregate advertising metrics by network and month
agg_spots_month = (
    spots_daily
    .groupby(['network_name', 'month'], as_index=False)
    .agg({
        'spend': 'sum',
        'impressions': 'sum',
        'lift': 'sum'
    })
)

# aggregate valid survey purchases by network and month
agg_survey_month = (
    survey[survey['valid_network']]
    .groupby(['network_name', 'month'], as_index=False)
    .agg({
        'purchases': 'sum'
    })
)

# merge advertising and survey data
monthly_metrics = agg_spots_month.merge(
    agg_survey_month,
    on=['network_name', 'month'],
    how='left'
)

# fill missing purchases with zero
monthly_metrics['purchases'] = monthly_metrics['purchases'].fillna(0)

In [ ]:
print("Monthly metrics shape:", monthly_metrics.shape)

In [ ]:
monthly_metrics.head()

In [ ]:
print(monthly_metrics.isnull().sum())

A quick validation step confirms that the aggregated dataset is structured correctly for downstream metric calculations.

### 6.2 Metric Calculation

With the aggregated dataset prepared, key performance metrics are calculated to evaluate the efficiency of TV advertising.

The following metrics are derived:

- CPM (Cost per Thousand Impressions)
- CPV (Cost per Visitor)
- Conversion Rate (CR)
- CPA (Cost per Acquisition)

All metrics are calculated using aggregated values to ensure accurate and interpretable results.

In [ ]:
import numpy as np

# Calculate Performance Metrics
monthly_metrics['CPM'] = monthly_metrics['spend'] / monthly_metrics['impressions'] * 1000

monthly_metrics['CPV'] = monthly_metrics['spend'] / monthly_metrics['lift'].replace(0, np.nan)

monthly_metrics['CR'] = monthly_metrics['purchases'] / monthly_metrics['lift'].replace(0, np.nan)

monthly_metrics['CPA'] = monthly_metrics['spend'] / monthly_metrics['purchases'].replace(0, np.nan)


In [ ]:
# Format Metrics
monthly_metrics[['CPM','CPV','CR','CPA']] = (
    monthly_metrics[['CPM','CPV','CR','CPA']].round(2)
)

In [ ]:
# Validation Checks
# shape check
print("Monthly metrics shape:", monthly_metrics.shape)

# preview data
display(monthly_metrics.head())

A small number of missing values are observed in CPV, CR, and CPA.

These occur when the denominator (lift or purchases) is zero, making the metric undefined.
Such cases are expected in real-world data and are retained as missing values to avoid misleading interpretations.

The key performance metrics (CPM, CPV, CR, and CPA) are calculated at the network and monthly level.

Basic validation checks confirm that the dataset has the expected structure and contains no missing values in the calculated metrics.

In [ ]:
print("\nMissing values (expected for zero denominators):")
print(monthly_metrics.isnull().sum())

### 6.3 Data Preparation for Visualization
To support downstream analysis and visualization, the aggregated dataset is further prepared to ensure compatibility with tools such as Tableau and to improve interpretability.

First, the month variable is converted from a period format to a string format, which is more suitable for visualization tools.
Next, missing values in derived metrics (CPV, CR, and CPA) are handled for presentation purposes. These missing values occur when the denominator (lift or purchases) is zero, resulting in undefined values. For visualization, these cases are replaced with zero to avoid gaps in charts while maintaining clarity.

Finally, the dataset is sorted by month and network name, and the index is reset to produce a clean and well-structured dataset for analysis and reporting.

In [ ]:
# Prepare Data for Visualization (Tableau-ready)

# convert month to string for visualization tools
monthly_metrics['month'] = monthly_metrics['month'].astype(str)

# handle missing values for presentation
monthly_metrics[['CPV', 'CR', 'CPA']] = (
    monthly_metrics[['CPV', 'CR', 'CPA']].fillna(0)
)

# sort for readability
monthly_metrics = monthly_metrics.sort_values(['month', 'network_name'])

# reset index
monthly_metrics = monthly_metrics.reset_index(drop=True)


# preview final dataset
display(monthly_metrics.head())

In [ ]:
print("Final dataset shape:", monthly_metrics.shape)

print("\nMissing values after visualization prep:")
print(monthly_metrics.isnull().sum())

### 6.4 Export Performance Dataset

In [ ]:
# export final dataset for Tableau
monthly_metrics.to_excel("media_performance_tableau_ready.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("media_performance_tableau_ready.xlsx")

# 7. Attribution Data Preparation

In addition to performance metrics, TV attribution requires a separate view of the data. While the performance dataset summarizes efficiency at the network and time-period level, attribution analysis depends on aligning individual TV airings with minute-level site traffic.

This section prepares a dedicated attribution dataset using spot-level airing data and site traffic data. The goal is to support a clear explanation of how TV airings may contribute to incremental site visits and to provide a foundation for a separate Tableau dashboard focused on attribution.

## 7.1 Objective

The objective of this section is to construct a dataset that links TV airings to changes in site traffic over time. Unlike the performance analysis, which focuses on aggregated efficiency metrics, attribution analysis looks at short-term traffic patterns around each airing.

By aligning airing timestamps with minute-level traffic and baseline estimates, this dataset will help illustrate whether TV spots are associated with observable increases in site visits. This provides additional context to validate the performance metrics and address potential skepticism around TV attribution.

## 7.2 Prepare Airing Data

The airing dataset contains spot-level information, including the network, program, and exact airing timestamps. For attribution analysis, the key requirement is to work with a consistent and precise time field that can be aligned with site traffic data.

In this step, the relevant timestamp field is selected and converted into a standardized datetime format. A simplified dataset is then created, retaining only the fields necessary for attribution analysis, such as network, airing time, and associated metrics. This ensures that the dataset is clean, focused, and ready for time-based joins with traffic data.

In [ ]:
# select relevant columns for attribution
airings = spots[['network_name', 'datetime/EST', 'spend', 'impressions', 'lift']].copy()

# convert airing time to datetime
airings['airing_time'] = pd.to_datetime(airings['datetime/EST'])

# standardize network name (consistent with earlier cleaning)
airings['network_name'] = airings['network_name'].str.lower().str.strip()

# keep only necessary columns
airings = airings[['network_name', 'airing_time', 'spend', 'impressions', 'lift']]

# preview
display(airings.head())

## 7.3 Prepare Traffic Data

The traffic dataset contains minute-level site activity along with a baseline estimate that represents expected traffic without TV influence. For attribution analysis, both actual traffic and baseline traffic are required to evaluate incremental impact.

In this step, the traffic timestamp is converted into a consistent datetime format to align with airing data. The dataset is then simplified to retain only the relevant fields, including timestamp, actual traffic, and baseline traffic. This ensures the data is structured for time-based analysis and comparison with airing events.

In [ ]:
traffic.columns

In [ ]:
# copy data
traffic_data = traffic.copy()

# convert time
traffic_data['timestamp'] = pd.to_datetime(traffic_data['datetime/EST'])

# keep only relevant metric (unique visitors)
traffic_data = traffic_data[traffic_data['metric'] == 'uvs']

# rename columns for clarity
traffic_data = traffic_data.rename(columns={
    'value': 'visitors'
})

# select needed columns
traffic_data = traffic_data[['timestamp', 'visitors', 'baseline']]

# sort
traffic_data = traffic_data.sort_values('timestamp').reset_index(drop=True)

# preview
display(traffic_data.head())

## 7.4 Build Attribution Dataset

To quantify the impact of TV advertising on site traffic, airing data is linked with traffic data based on time proximity.

For each TV airing, a short time window is defined following the airing timestamp to capture potential traffic responses. Within this window, observed traffic is compared against baseline traffic to estimate incremental visits driven by TV exposure.

This approach reflects how TV advertising may generate immediate spikes in site activity, allowing for a more intuitive validation of attribution assumptions.

The result is an attribution dataset that connects each airing event with corresponding traffic behavior, forming the basis for further analysis and visualization.

In [ ]:
# define attribution window (e.g., 5 minutes after airing)
window_minutes = 5

In [ ]:
# expand airing time window
airings['window_end'] = airings['airing_time'] + pd.Timedelta(minutes=window_minutes)

# perform merge (cross join style via filtering)
attribution_rows = []

for _, airing in airings.iterrows():
    temp = traffic_data[
        (traffic_data['timestamp'] >= airing['airing_time']) &
        (traffic_data['timestamp'] <= airing['window_end'])
    ].copy()

    temp['network_name'] = airing['network_name']
    temp['airing_time'] = airing['airing_time']

    attribution_rows.append(temp)

# combine all
attribution_data = pd.concat(attribution_rows, ignore_index=True)

In [ ]:
# calculate incremental traffic
attribution_data['incremental_visitors'] = (
    attribution_data['visitors'] - attribution_data['baseline']
)

# remove negative noise (optional but recommended)
attribution_data['incremental_visitors'] = attribution_data['incremental_visitors'].clip(lower=0)

In [ ]:
display(attribution_data.head())

print("Attribution dataset shape:", attribution_data.shape)

## 7.5 Export Attribution Dataset

The attribution dataset is exported for visualization and analysis in Tableau.

This dataset captures the relationship between TV airings and subsequent site traffic behavior, including estimated incremental visitors. It serves as the foundation for validating TV attribution and building interactive dashboards.

In [ ]:
# export attribution dataset
attribution_data.to_excel("media_attribution_tableau_ready.xlsx", index=False)

# download (for Colab)
from google.colab import files
files.download("media_attribution_tableau_ready.xlsx")

## 8. Summary of Findings
Overall, TV advertising drives a clear and measurable impact on website traffic and conversions, though performance varies significantly across networks and over time.

First, traffic response occurs immediately after airing. Website visits exceed baseline levels right after a TV spot airs, with a noticeable peak within the first minute, followed by a gradual decay. This confirms a strong short-term causal relationship between TV exposure and online engagement.

Second, contribution is highly concentrated across networks. A small number of networks—such as Network A, Network B, and Network C—account for a disproportionately large share of incremental traffic, suggesting that not all channels are equally effective at driving demand.

Third, efficiency and scale are not always aligned. While some networks deliver high conversion rates or low CPA, others generate large volumes of conversions but at a higher cost. However, several networks successfully achieve both scale and efficiency, indicating strong candidates for budget prioritization.

Fourth, performance trends over time suggest improving efficiency. From April to May, both spend and conversions increased, while CPA decreased slightly, indicating better cost efficiency. However, June data appears incomplete and should not be used for decision-making.

Finally, variability across airings is significant. Even within the same network, performance can vary widely, highlighting the importance of considering consistency—not just averages—when evaluating channel effectiveness.

## 9. Next Steps
Based on these findings, several actions can be taken to improve TV advertising effectiveness:

1. Reallocate budget toward high-performing networks  
Prioritize networks that consistently deliver both strong conversion rates and low CPA, such as Network A and Network B.

2. Reduce or reassess inefficient channels  
Identify networks with high CPA and low conversion efficiency, and consider reducing spend or testing alternative placements.

3. Incorporate consistency into decision-making  
Beyond average performance, evaluate variability across airings to prioritize networks with stable and predictable outcomes.

4. Improve measurement through integrated attribution  
Combine survey-based attribution with traffic-based validation (as shown in this analysis) to obtain a more robust understanding of TV impact.

5. Refine data collection and attribution design  
Address missing or “other” network responses and ensure more complete tracking of June data to improve reliability of future analysis.

6. Expand analysis across time and granularity  
Further analyze performance by week, time-of-day, and program type to uncover more granular optimization opportunities.